# Week 4 — Date Functions: Reading Time Out of Text
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Wednesday you invented a column out of a rule. Today you invent one out of a *string*. Every timestamp in `orders` is stored as ordinary text — `2017-11-24 09:22:16` — and "November", "2017" and "twelve days later" are all buried inside it. `strftime()` carves a piece out of that text so you can group by it; `julianday()` turns two of those texts into a number of days so you can measure a duration. Between them, the five timestamp columns in `orders` stop being opaque and become the operational history of the business.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom. The check cells read the exact column aliases each question asks for, so use the alias names as written.

Two things will catch you today, and neither of them raises an error:

- `strftime()` returns **TEXT**. `WHERE strftime('%Y', order_purchase_timestamp) = 2017` compares a string against a number, matches nothing, and cheerfully reports `0`. Quote the year: `= '2017'`.
- Subtracting two timestamps directly does **not** fail — SQLite reads digits off the front of each string, so a 12.6-day average silently collapses to a fraction of a day. Wrap both sides in `julianday()` before subtracting.

🤖 **Using DeepSeek this week:** you may ask DeepSeek to help draft these queries, but the prompt-then-verify protocol from the demo still applies — tell it the table, the exact columns and that the dialect is **SQLite** (`strftime` and `julianday` are SQLite functions; ask for MySQL or Postgres and you will get `YEAR()` and `DATEDIFF()`, which do not exist here). Then **run the query and check the number against the Expected value before you trust it**. Date bugs are the quietest bugs in SQL: they return a tidy table with a plausible wrong number in it. The check cells are your verification step; never edit one to make a wrong query pass.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Question 1 — Is the business growing?

The single most-asked question about any transactional table is *"how are we trending?"*, and `orders` cannot answer it as stored: there is no `year` column. There is only `order_purchase_timestamp`, with the year sitting in its first four characters.

Write a query against `orders` that returns **one row per year**, with two columns:

- `year` — `strftime('%Y', order_purchase_timestamp)`, aliased
- `order_count` — `COUNT(*)`

Group by the alias you just created and sort chronologically.

When you read the result, treat 2016 with suspicion. The dataset begins in **September 2016**, so that row covers roughly four months, not twelve — quoting it beside 2017 and 2018 as if it were a full year is the kind of mistake that makes it into a board pack.

**Expected:** three rows — 2016 **329**, 2017 **45,101**, 2018 **54,011**. Those three add up to all **99,441** orders in the table.

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
for col in ['year', 'order_count']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT aliases"
assert q1.shape[0] == 3, \
    f"Q1: expected 3 year rows, got {q1.shape[0]} — group by the extracted year, not the timestamp"

years = {str(r['year']): int(r['order_count']) for _, r in q1.iterrows()}
for y, n in [('2016', 329), ('2017', 45101), ('2018', 54011)]:
    assert y in years, f"Q1: no row for {y} — is your format code '%Y' (capital Y)?"
    assert years[y] == n, f"Q1: expected {n:,} orders in {y}, got {years[y]:,}"
assert sum(years.values()) == 99441, \
    f"Q1: the three years should account for all 99,441 orders, got {sum(years.values()):,}"

labels = [str(v) for v in q1['year']]
assert labels == sorted(labels), \
    "Q1: rows are not in chronological order — add ORDER BY year"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Find the peak month

A year is too coarse to staff a warehouse on. Zoom in to 2017 and look at it month by month.

Write a query against `orders` that returns **one row per month of 2017**, with two columns:

- `month` — `strftime('%Y-%m', order_purchase_timestamp)`, aliased
- `order_count` — `COUNT(*)`

Filter to 2017 in the `WHERE` clause, group by `month`, and sort chronologically.

Two details decide whether this works. First, use `'%Y-%m'`, not `'%m'` — grouping on the month alone would merge November 2017 and November 2018 into one bucket and destroy the trend. Second, mind the **quotes** in the `WHERE` clause: `strftime()` hands back TEXT, so the comparison value is `'2017'`, not `2017`.

**Expected:** 12 rows — 2017-01 **800**, 2017-02 **1,780**, 2017-03 **2,682**, 2017-04 **2,404**, 2017-05 **3,700**, 2017-06 **3,245**, 2017-07 **4,026**, 2017-08 **4,331**, 2017-09 **4,285**, 2017-10 **4,631**, 2017-11 **7,544**, 2017-12 **5,673**. They sum to the **45,101** you found for 2017 in Question 1.

Look at November before you move on: **7,544 orders**, nearly double a typical month. That is Black Friday, and it is the single most important fact about Olist's calendar.

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
for col in ['month', 'order_count']:
    assert col in q2.columns, f"Q2: missing the '{col}' column — check your SELECT aliases"
assert q2.shape[0] == 12, \
    (f"Q2: expected 12 monthly rows, got {q2.shape[0]} — if you got 0, the WHERE value needs "
     f"QUOTES ('2017'); if you got more, the filter is not restricting to 2017")

expected = {'2017-01': 800,  '2017-02': 1780, '2017-03': 2682, '2017-04': 2404,
            '2017-05': 3700, '2017-06': 3245, '2017-07': 4026, '2017-08': 4331,
            '2017-09': 4285, '2017-10': 4631, '2017-11': 7544, '2017-12': 5673}
got = {str(r['month']): int(r['order_count']) for _, r in q2.iterrows()}
for m in sorted(expected):
    assert m in got, f"Q2: no row for {m} — is your format code '%Y-%m'?"
    assert got[m] == expected[m], f"Q2: expected {expected[m]:,} orders in {m}, got {got[m]:,}"
assert sum(got.values()) == 45101, \
    f"Q2: the 12 months should sum to the 45,101 orders of 2017, got {sum(got.values()):,}"

labels = [str(v) for v in q2['month']]
assert labels == sorted(labels), \
    "Q2: rows are not in chronological order — add ORDER BY month"
assert max(got, key=got.get) == '2017-11', "Q2: the peak month should be 2017-11 (Black Friday)"
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — How long does delivery actually take?

`strftime()` answers *when*. The other half of date work is *how long*, and that means subtracting one timestamp from another — which you cannot do directly, because both are text.

`julianday()` is the bridge: it converts a timestamp into a single number (days elapsed since a fixed point far back in history), so the difference between two of them is plain arithmetic. Because the conversion keeps the time of day, the result is fractional.

Write a query against `orders` returning a **single row, single column**:

- `avg_delivery_days` — the average of `julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)`, wrapped in `ROUND(..., 1)`

Restrict the query to orders that were actually delivered (`order_status = 'delivered'`) **and** whose `order_delivered_customer_date` `IS NOT NULL` — a handful of delivered orders are missing that timestamp, and `IS NOT NULL` is the only way to test for it (`= NULL` is never true).

**Expected:** one row — **12.6** days.

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert q3.shape[0] == 1, f"Q3: expected a single row, got {q3.shape[0]} — no GROUP BY is needed here"
assert 'avg_delivery_days' in q3.columns, \
    "Q3: missing the 'avg_delivery_days' column — check your SELECT alias"
got = round(float(q3.iloc[0]['avg_delivery_days']), 1)
assert abs(got - 12.6) < 0.05, \
    (f"Q3: expected 12.6 average delivery days, got {got}. A number far below 1 means you "
     f"subtracted the timestamps as text — wrap BOTH in julianday() before subtracting. A "
     f"negative number means the two dates are the wrong way round.")
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — Promise versus reality

`orders` holds a column most e-commerce datasets do not: `order_estimated_delivery_date`, the date the customer was shown at checkout. Sitting beside `order_delivered_customer_date`, it lets you measure something no average can capture — how often the business broke its own promise.

The comparison needs no function at all. Both columns are ISO-formatted text, and ISO text sorts in chronological order, so `order_delivered_customer_date > order_estimated_delivery_date` already means "arrived after the promised date".

Write a query against `orders` returning a **single row** with two columns:

- `late_orders` — `COUNT(*)` of delivered orders that arrived after their estimated date
- `pct_of_delivered` — those late orders as a percentage of **all** delivered orders, rounded to 1 decimal place

The denominator is the trap. It is not `COUNT(*)` of the rows this query is looking at — those are only the late ones. Pull it from a subquery over the whole table: `(SELECT COUNT(*) FROM orders WHERE order_status = 'delivered')`. And force REAL division with `* 1.0`, or SQLite will truncate the share to a whole number.

**Expected:** one row — **7,826** late orders, **8.1%** of delivered orders.

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert q4.shape[0] == 1, f"Q4: expected a single row, got {q4.shape[0]}"
for col in ['late_orders', 'pct_of_delivered']:
    assert col in q4.columns, f"Q4: missing the '{col}' column — check your SELECT aliases"
assert int(q4.iloc[0]['late_orders']) == 7826, \
    (f"Q4: expected 7,826 late orders, got {int(q4.iloc[0]['late_orders']):,} — the filter is "
     f"delivered_customer_date > estimated_delivery_date AND order_status = 'delivered'")
pct = float(q4.iloc[0]['pct_of_delivered'])
assert abs(pct - 8.1) < 0.05, \
    (f"Q4: expected 8.1% of delivered orders, got {pct} — a whole number means integer "
     f"division truncated it (multiply by 1.0); a much larger number means the denominator "
     f"is not all 96,478 delivered orders")
print("✅ Q4 correct")
q4  # show the result of your query

## Question 5 — The two halves of Week 4 in one row

Question 1 gives you the yearly trend as three rows. A scorecard wants it as **one row with three columns**, because that is the shape you put on a slide next to last quarter's version.

You already have the tool: Wednesday's `SUM(CASE WHEN <condition> THEN 1 ELSE 0 END)`, which scores every row 1 or 0 so each column carries its own filter. The only new part is that the condition is now a `strftime()` expression rather than a stored column.

Write a query against `orders` returning a **single row** with four columns:

- `orders_2016` — count of orders whose purchase year is 2016
- `orders_2017` — same for 2017
- `orders_2018` — same for 2018
- `total_orders` — `COUNT(*)` over the whole table

No `WHERE` and no `GROUP BY` — every condition lives inside its own `CASE`. Remember the quotes: you are comparing `strftime('%Y', ...)` against `'2016'`, a string.

**Expected:** one row — 329 | 45,101 | 54,011 | 99,441. The three year columns must add up to the total; if they do not, a year is being missed by the `CASE` conditions.

In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert q5.shape[0] == 1, f"Q5: expected a single row, got {q5.shape[0]} — no GROUP BY is needed here"
for col, n in [('orders_2016', 329), ('orders_2017', 45101),
               ('orders_2018', 54011), ('total_orders', 99441)]:
    assert col in q5.columns, f"Q5: missing the '{col}' column — check your SELECT aliases"
    assert int(q5.iloc[0][col]) == n, \
        (f"Q5: expected {col} = {n:,}, got {int(q5.iloc[0][col]):,} — a 0 usually means the "
         f"year was compared as a NUMBER; strftime() returns TEXT, so quote it")
years_total = sum(int(q5.iloc[0][c]) for c in ['orders_2016', 'orders_2017', 'orders_2018'])
assert years_total == int(q5.iloc[0]['total_orders']), \
    (f"Q5: the three year columns sum to {years_total:,} but total_orders is "
     f"{int(q5.iloc[0]['total_orders']):,} — every order should land in exactly one year")
print("✅ Q5 correct")
q5  # show the result of your query